# 2. Tabellen voor de potentiële windsnelheid

Python-versie van het R-script **`production-wind10-impr.Rmd`** (KNMI, versie februari 2026).

Dit notebook maakt de tabellen van de potentiële windsnelheid (voor het BOI) uit de resultaten van
`1_statistics_wind.ipynb`:

1. de verdeling van de windrichting bij Hoek van Holland uit SEAS5 berekenen. Deze tabel wordt voor alle
   stations gebruikt, ook in `6_production_wl.ipynb`;
2. per station en richtingssector: een gewone 2-parameter Weibull-staart opnieuw fitten en de tabel met
   voorwaardelijke kansen berekenen, voor sectoren van 22,5 graden en van 30 graden.

**Let op:** het R-script maakt de tabellen met de richtingsverdeling uit het invoerbestand
`hoekvanh_winddirs.RData` (`Table22h`, `Table30`), niet met de verdeling die het zelf berekent. Dit notebook
gebruikt wel de zelf berekende verdeling (de csv-bestanden van stap 1). De twee verschillen 0,0001 in de
grootste sector (225 graden en 240 graden); in die sector kan het laatste cijfer van de tabellen daardoor
afwijken van de tabellen van KNMI.

In dit notebook worden geen toevalsgetallen gebruikt.

**Gebruik:** voer eerst `1_statistics_wind.ipynb` uit, controleer de instellingen hieronder en voer daarna alle cellen uit.

Uitvoer (dezelfde bestanden als het R-script, in `output_path`):
- `<station>_2023_potwind_condprob_12hour_{22hdeg,30deg}_versionFeb2026.csv`
- `<station>_2023_potwind_Wbltails_{22hdeg,30deg}_versionFeb2026.csv`
- `<station>_2023_potwind_stats_refit_22hdeg_versionFeb2026.pkl`, `<station>_2023_potwind_stats_wind_refit_30deg_versionFeb2026.pkl`
- `hoekvanh_2035_windrichting_{22hdeg,30deg}_versionFeb2026.csv`

## Instellingen
De standaardwaarden van de mappen staan in `kuststatistiek/settings.py`.

In [ ]:
from kuststatistiek import settings

input_path = settings.DATA_PATH  # map met de aangeleverde invoerdata
output_path = (
    settings.OUTPUT_PATH
)  # map met de resultaten van notebook 1; de tabellen komen hier ook

## Voorbereiding
Laad de instellingen en het model voor de extremale index die notebook 1 heeft weggeschreven.
De stations zijn die van de R-run; stations die in notebook 1 niet zijn doorgerekend, worden overgeslagen.

In [ ]:
from kuststatistiek.settings import Paths, ProductionBins
from kuststatistiek.io.read_wind import read_seas5_wind
from kuststatistiek.production import (
    direction_distribution,
    read_direction_table,
    weibull_refit_wind,
)
from kuststatistiek.rutils import load_pkl

paths = Paths(input_path=input_path, output_path=output_path)
s = load_pkl(paths.out("statistics-wind10settings.pkl"))
eimdl = load_pkl(paths.out("eimdl.pkl"))["eimdl"]
stations = s["stationsys5"]
pbins = ProductionBins()  # sectoren van 22,5 en 30 graden voor de tabellen
print("stations:", stations)

## Stap 1 – Verdeling van de windrichting bij Hoek van Holland (R-chunk `eiseas5`)
Het deel van de tijd dat de wind uit elke sector komt, berekend uit de richting van de SEAS5-windschuifspanning.
Deze bestanden worden hieronder in stap 2 en in `6_production_wl.ipynb` gebruikt.

In [ ]:
dstress = read_seas5_wind("hoekvanh", paths.input_path)["dstress"]
direction_distribution(dstress, pbins, paths)

## Stap 2 – Weibull-refit en tabellen (R-chunks `wblfit` en `wblfit30`)
Eerst voor sectoren van 22,5 graden, daarna voor sectoren van 30 graden, met de richtingsverdeling uit stap 1.

In [ ]:
ptab22 = read_direction_table(
    paths.out("hoekvanh_2035_windrichting_22hdeg_versionFeb2026.csv")
)
ptab30 = read_direction_table(
    paths.out("hoekvanh_2035_windrichting_30deg_versionFeb2026.csv")
)

for dirbins, ptab in [("22h", ptab22), ("30", ptab30)]:
    for station in stations:
        f = paths.out(f"{station}_stats_wind_boot2.pkl")
        if not f.exists():
            print(
                f"{station}: {f.name} niet gevonden, overgeslagen (voer eerst notebook 1 uit voor dit station)"
            )
            continue
        boot2 = load_pkl(f)
        weibull_refit_wind(
            station,
            boot2,
            eimdl,
            s["EIvalue"],
            s["factor"],
            s["freq"],
            pbins,
            ptab,
            dirbins,
            paths,
            s["resolutionfactor"],
        )
        print(f"{station}: windtabellen {dirbins} graden klaar")